# OmniSpeak (Colab)

Đọc văn bản, nhân bản giọng nói, lưu thư viện giọng — chạy trên Google Colab, dùng model [OmniVoice](https://github.com/k2-fsa/OmniVoice) (`k2-fsa/OmniVoice`, Apache-2.0).

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy các cell theo thứ tự từ trên xuống — mọi cell đều an toàn khi chạy lại.


## Cài đặt

In [ ]:
# 1. GPU check
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Không có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")


In [ ]:
# 2. Cài đặt
import subprocess
import sys

def run(cmd, what=""):
    print("\n$", cmd if isinstance(cmd, str) else " ".join(cmd))
    if subprocess.run(cmd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"Lỗi: {what or cmd}. Xem log phía trên rồi chạy lại cell này.")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1")
run([sys.executable, "-m", "pip", "install", "-q",
     "omnivoice", "fastapi", "uvicorn[standard]", "python-multipart", "soundfile"])
run([sys.executable, "-c",
     "from omnivoice import OmniVoice, VoiceClonePrompt; import fastapi, soundfile; print('OK')"])


In [ ]:
# 3. Giao diện web
import os

FRONTEND_DIR = "/content/omnispeak_frontend"
os.makedirs(FRONTEND_DIR, exist_ok=True)

UI_HTML = '<!DOCTYPE html>\n<html lang="vi">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>OmniSpeak (Colab)</title>\n<style>\n  :root{\n    --purple:#6C5CE7; --purple-dark:#5b4bd6; --bg:#f4f5fb; --card:#ffffff;\n    --border:#e4e4ef; --text:#1f2030; --muted:#8a8ca0; --radius:14px;\n  }\n  body.dark{\n    --bg:#0f172a; --card:#1e293b; --border:#334155; --text:#f1f5f9; --muted:#94a3b8;\n  }\n  *{box-sizing:border-box}\n  body{margin:0;font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif;background:var(--bg);color:var(--text);transition:background .2s,color .2s}\n  .layout{display:flex;min-height:100vh}\n  .sidebar{width:230px;background:var(--card);border-right:1px solid var(--border);padding:24px 14px;display:flex;flex-direction:column;gap:6px;flex-shrink:0}\n  .brand{display:flex;align-items:center;gap:10px;padding:0 10px 22px 10px;font-weight:700;font-size:18px}\n  .brand .dot{width:34px;height:34px;border-radius:10px;background:var(--purple);display:flex;align-items:center;justify-content:center;color:#fff}\n  .navitem{display:flex;align-items:center;gap:10px;padding:11px 14px;border-radius:10px;cursor:pointer;color:var(--muted);font-size:14px;font-weight:500}\n  .navitem:hover{background:color-mix(in srgb, var(--purple) 8%, transparent)}\n  .navitem.active{background:var(--purple);color:#fff}\n  .main{flex:1;padding:32px 40px 120px 40px;position:relative;min-width:0}\n  .topbar{display:flex;justify-content:flex-end;margin-bottom:8px}\n  .theme-toggle{background:var(--card);border:1px solid var(--border);border-radius:10px;width:40px;height:40px;font-size:18px;cursor:pointer;display:flex;align-items:center;justify-content:center;color:var(--text)}\n  .theme-toggle:hover{background:color-mix(in srgb, var(--purple) 10%, var(--card))}\n  h1{font-size:24px;margin:0 0 4px 0}\n  .sub{color:var(--muted);font-size:14px;margin:0 0 24px 0}\n  .card{background:var(--card);border:1px solid var(--border);border-radius:var(--radius);padding:22px;margin-bottom:18px}\n  textarea, input[type=text]{width:100%;border:1px solid var(--border);border-radius:10px;padding:12px 14px;font-size:14px;font-family:inherit;resize:vertical;background:var(--card);color:var(--text)}\n  textarea{min-height:220px;max-height:420px;overflow-y:auto}\n  label{font-size:13px;font-weight:600;color:var(--muted);display:block;margin-bottom:6px;text-transform:uppercase;letter-spacing:.02em}\n  .row{display:flex;gap:12px;margin-top:14px;flex-wrap:wrap;align-items:center}\n  .row.between{justify-content:space-between}\n  .btn{background:var(--purple);color:#fff;border:none;padding:12px 22px;border-radius:10px;font-size:14px;font-weight:600;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}\n  .btn:hover{background:var(--purple-dark)}\n  .btn.secondary{background:color-mix(in srgb, var(--purple) 12%, var(--card));color:var(--purple)}\n  .btn.danger{background:#fde8e8;color:#e5484d}\n  .btn:disabled{opacity:.5;cursor:not-allowed}\n  audio{width:100%;margin-top:14px}\n  .voice-item{display:flex;justify-content:space-between;align-items:center;padding:12px 14px;border:1px solid var(--border);border-radius:10px;margin-bottom:8px;gap:12px}\n  .voice-item .name{font-weight:600;font-size:14px}\n  .voice-item .meta{color:var(--muted);font-size:12px}\n  .voice-item .actions{display:flex;gap:8px;flex-shrink:0}\n  .preview-btn{width:30px;height:30px;border-radius:50%;border:1px solid var(--border);background:var(--card);color:var(--purple);cursor:pointer;display:flex;align-items:center;justify-content:center;font-size:11px;flex-shrink:0}\n  .preview-btn:hover{background:color-mix(in srgb, var(--purple) 10%, var(--card))}\n  .preview-btn.playing{background:var(--purple);color:#fff}\n  .hist-item{padding:12px 14px;border:1px solid var(--border);border-radius:10px;margin-bottom:8px}\n  .hist-item .text{font-size:14px;margin-bottom:6px}\n  .hist-item .meta{color:var(--muted);font-size:12px;margin-bottom:8px}\n  .status{font-size:13px;color:var(--muted);margin-top:10px;min-height:18px}\n  .status.err{color:#e5484d}\n  .status.ok{color:#12b76a}\n  .status.warn{color:#d68a00}\n  .tab-section{display:none}\n  .tab-section.active{display:block}\n  .rec-dot{width:10px;height:10px;border-radius:50%;background:#e5484d;display:inline-block;margin-right:6px;animation:pulse 1s infinite}\n  @keyframes pulse{0%,100%{opacity:1}50%{opacity:.3}}\n  .empty{color:var(--muted);font-size:14px;text-align:center;padding:24px}\n  .refresh-link{font-size:12px;color:var(--purple);cursor:pointer;text-decoration:underline}\n  .counts{color:var(--muted);font-size:12px}\n\n  /* --- TTS two-column layout --- */\n  .tts-columns{display:flex;gap:18px;align-items:flex-start}\n  .tts-left{flex:1;min-width:0}\n  .tts-right{width:320px;flex-shrink:0}\n  @media (max-width:900px){ .tts-columns{flex-direction:column} .tts-right{width:100%} }\n\n  /* --- Segmented tabs (Giọng nói / Lịch sử) --- */\n  .segmented{display:flex;background:color-mix(in srgb, var(--muted) 12%, transparent);border-radius:10px;padding:4px;margin-bottom:14px}\n  .seg-btn{flex:1;text-align:center;padding:8px 10px;border-radius:8px;font-size:13px;font-weight:600;color:var(--muted);cursor:pointer;background:transparent;border:none}\n  .seg-btn.active{background:var(--card);color:var(--text);box-shadow:0 1px 3px rgba(0,0,0,.08)}\n\n  .search-box{position:relative;margin-bottom:12px}\n  .search-box input{padding-left:34px}\n  .search-box .icon{position:absolute;left:12px;top:50%;transform:translateY(-50%);color:var(--muted);font-size:13px}\n\n  .voice-row{display:flex;align-items:center;gap:10px;padding:10px 12px;border-radius:10px;cursor:pointer;border:1px solid transparent;margin-bottom:4px}\n  .voice-row:hover{background:color-mix(in srgb, var(--purple) 6%, transparent)}\n  .voice-row.selected{background:color-mix(in srgb, var(--purple) 14%, transparent);border-color:var(--purple)}\n  .voice-row .play-ic{width:26px;height:26px;border-radius:50%;background:color-mix(in srgb, var(--purple) 16%, var(--card));color:var(--purple);display:flex;align-items:center;justify-content:center;font-size:11px;flex-shrink:0}\n  .voice-row.selected .play-ic{background:var(--purple);color:#fff}\n  .voice-row .vname{font-weight:600;font-size:13px}\n  .voice-row .vmeta{font-size:11px;color:var(--muted)}\n  .hist-row{padding:10px 12px;border-radius:10px;cursor:pointer;margin-bottom:4px}\n  .hist-row:hover{background:color-mix(in srgb, var(--purple) 6%, transparent)}\n  .hist-row .htext{font-size:13px;font-weight:600;margin-bottom:2px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\n  .hist-row .hmeta{font-size:11px;color:var(--muted)}\n\n  /* --- Bottom player bar --- */\n  .player-bar{position:fixed;left:230px;right:0;bottom:0;background:var(--card);border-top:1px solid var(--border);padding:10px 24px;display:none;align-items:center;gap:16px;z-index:50}\n  @media (max-width:900px){ .player-bar{left:0} }\n  .player-seek{position:absolute;top:-1px;left:0;right:0;width:100%;height:3px;-webkit-appearance:none;appearance:none;background:var(--border);cursor:pointer;margin:0}\n  .player-seek::-webkit-slider-thumb{-webkit-appearance:none;width:0;height:0}\n  .player-seek::-moz-range-thumb{width:0;height:0;border:none}\n  .player-seek::-webkit-slider-runnable-track{background:linear-gradient(to right, var(--purple) var(--pct,0%), var(--border) 0%)}\n  .player-icon{width:40px;height:40px;border-radius:10px;background:color-mix(in srgb, var(--purple) 16%, var(--card));color:var(--purple);display:flex;align-items:center;justify-content:center;font-size:16px;flex-shrink:0}\n  .player-info{min-width:120px}\n  .player-info .pname{font-weight:700;font-size:13px}\n  .player-info .pmeta{font-size:11px;color:var(--muted)}\n  .player-time{font-size:12px;color:var(--muted);width:96px;text-align:right;flex-shrink:0}\n  .player-controls{display:flex;align-items:center;gap:6px;margin-left:auto;flex-shrink:0}\n  .pbtn{width:36px;height:36px;border-radius:8px;border:1px solid var(--border);background:var(--card);color:var(--text);cursor:pointer;display:flex;align-items:center;justify-content:center;font-size:14px}\n  .pbtn:hover{background:color-mix(in srgb, var(--purple) 8%, var(--card))}\n  .pbtn.play{background:var(--purple);color:#fff;border-color:var(--purple)}\n  .pbtn.play:hover{background:var(--purple-dark)}\n  .pbtn.speed{width:auto;padding:0 10px;font-size:12px;font-weight:700}\n\n  .conn-banner{position:fixed;top:0;left:230px;right:0;background:#e5484d;color:#fff;font-size:13px;font-weight:600;text-align:center;padding:8px;display:none;z-index:60}\n  @media (max-width:900px){ .conn-banner{left:0} }\n</style>\n</head>\n<body>\n<div class="conn-banner" id="connBanner">⚠️ Mất kết nối tới backend — đang thử kết nối lại...</div>\n<div class="layout">\n  <div class="sidebar">\n    <div class="brand"><div class="dot">🎙️</div>OmniSpeak</div>\n    <div class="navitem active" data-tab="tts">🔊 Đọc văn bản</div>\n    <div class="navitem" data-tab="library">📚 Thư viện giọng nói</div>\n    <div class="navitem" data-tab="voices">🎤 Giọng nói của bạn</div>\n    <div class="navitem" data-tab="history">🕘 Lịch sử</div>\n  </div>\n  <div class="main">\n    <div class="topbar">\n      <button class="theme-toggle" id="themeToggle" onclick="toggleTheme()" title="Chuyển chế độ sáng/tối">🌙</button>\n    </div>\n\n    <!-- TTS TAB -->\n    <div class="tab-section active" id="tab-tts">\n      <h1>Đọc văn bản</h1>\n      <p class="sub">Chuyển văn bản thành giọng nói bằng OmniVoice, chạy local trên Colab.</p>\n\n      <div class="tts-columns">\n        <div class="tts-left">\n          <div class="card">\n            <label>Văn bản cần đọc</label>\n            <textarea id="ttsText" placeholder="Nhập văn bản cần đọc..." oninput="updateCounts()"></textarea>\n            <div class="row between">\n              <span class="counts" id="charCount">0 ký tự</span>\n              <span class="counts" id="wordCount">0 từ</span>\n            </div>\n            <div class="status" id="lengthWarning"></div>\n            <div class="row between">\n              <span class="counts">Giọng: <strong id="selectedVoiceName">Mặc định</strong></span>\n              <button class="btn" id="genBtn" onclick="generateTTS()">▷ Tạo giọng đọc</button>\n            </div>\n            <div class="status" id="ttsStatus"></div>\n          </div>\n        </div>\n\n        <div class="tts-right">\n          <div class="card">\n            <div class="segmented">\n              <button class="seg-btn active" id="segVoices" onclick="switchRightTab(\'voices\')">Giọng nói</button>\n              <button class="seg-btn" id="segHistory" onclick="switchRightTab(\'history\')">Lịch sử</button>\n            </div>\n\n            <div id="rightPanel-voices">\n              <div class="search-box">\n                <span class="icon">🔍</span>\n                <input type="text" id="voiceSearch" placeholder="Tìm giọng nói..." oninput="renderVoicePicker()">\n              </div>\n              <div id="voicePickerList"></div>\n            </div>\n\n            <div id="rightPanel-history" style="display:none">\n              <div id="historyPanelList"><div class="empty">Chưa có lịch sử nào.</div></div>\n            </div>\n          </div>\n        </div>\n      </div>\n    </div>\n\n    <!-- LIBRARY TAB -->\n    <div class="tab-section" id="tab-library">\n      <h1>Thư viện giọng nói</h1>\n      <p class="sub">Toàn bộ giọng bạn đã lưu — dùng lại ở tab "Đọc văn bản" hoặc xoá nếu không cần nữa. <span class="refresh-link" onclick="loadVoices()">Làm mới</span></p>\n      <div class="card">\n        <div id="voiceList"><div class="empty">Chưa có giọng nào — sang tab "Giọng nói của bạn" để tạo giọng đầu tiên.</div></div>\n      </div>\n    </div>\n\n    <!-- VOICES TAB (create/record) -->\n    <div class="tab-section" id="tab-voices">\n      <h1>Giọng nói của bạn</h1>\n      <p class="sub">Ghi âm hoặc upload một đoạn mẫu (3–10 giây) để nhân bản giọng — giọng lưu xong sẽ hiện ở tab "Thư viện giọng nói".</p>\n      <div class="card">\n        <label>Tên giọng nói</label>\n        <input type="text" id="voiceName" placeholder="VD: Giọng của tôi" maxlength="80">\n        <div class="row">\n          <button class="btn secondary" id="recBtn" onclick="toggleRecord()">🎙️ Bắt đầu ghi âm</button>\n          <input type="file" id="fileInput" accept="audio/*" style="display:none" onchange="onFileChosen()">\n          <button class="btn secondary" onclick="document.getElementById(\'fileInput\').click()">📁 Upload file</button>\n        </div>\n        <div class="status" id="recStatus"></div>\n        <audio id="refPreview" controls style="display:none;margin-top:12px"></audio>\n        <div class="row">\n          <button class="btn" id="saveVoiceBtn" onclick="saveVoiceProfile()" disabled>Lưu vào thư viện</button>\n        </div>\n        <div class="status" id="voiceStatus"></div>\n      </div>\n    </div>\n\n    <!-- HISTORY TAB -->\n    <div class="tab-section" id="tab-history">\n      <h1>Lịch sử</h1>\n      <p class="sub">Các đoạn đã tạo trong phiên làm việc này (mất khi tải lại trang hoặc Colab ngắt session).</p>\n      <div class="card">\n        <div id="historyList"><div class="empty">Chưa có lịch sử nào.</div></div>\n      </div>\n    </div>\n\n  </div>\n</div>\n\n<!-- Bottom player bar -->\n<div class="player-bar" id="playerBar">\n  <input type="range" class="player-seek" id="playerSeek" min="0" max="100" value="0" oninput="seekPlayer()">\n  <div class="player-icon">🎵</div>\n  <div class="player-info">\n    <div class="pname" id="playerName">—</div>\n    <div class="pmeta" id="playerMeta">—</div>\n  </div>\n  <div class="player-time" id="playerTime">0:00 / 0:00</div>\n  <div class="player-controls">\n    <button class="pbtn play" id="playPauseBtn" onclick="togglePlayPause()" title="Phát/Tạm dừng">▷</button>\n    <button class="pbtn" onclick="stopPlayer()" title="Dừng">⏹</button>\n    <button class="pbtn speed" id="speedBtn" onclick="cycleSpeed()" title="Tốc độ">1x</button>\n    <a class="pbtn" id="playerDownload" download title="Tải xuống">⬇️</a>\n    <button class="pbtn" id="muteBtn" onclick="toggleMute()" title="Tắt/Bật tiếng">🔊</button>\n    <button class="pbtn" onclick="closePlayer()" title="Đóng">✕</button>\n  </div>\n  <audio id="playerAudio" style="display:none"></audio>\n</div>\n<audio id="libPreviewAudio" style="display:none"></audio>\n\n<script>\nconst BASE = window.location.origin;\nlet mediaRecorder, recordedChunks = [], recordedBlob = null, isRecording = false;\nlet historyItems = [];\nlet allProfiles = [];\nlet selectedProfileId = "";\nlet selectedProfileName = "Mặc định";\n\n// ---- Giới hạn độ dài văn bản ----\nconst SOFT_WORD_LIMIT = 2000;  // vẫn tạo được, chỉ cảnh báo sẽ xử lý theo từng đoạn + mất thời gian\nconst HARD_WORD_LIMIT = 3000;  // vượt mức này thì chặn, yêu cầu rút gọn\nconst MAX_PROFILES = 20;       // khớp với MAX_PROFILES ở backend.py — chỉ để hiển thị, backend là nơi chặn thật\n\n// ---- Theme (mặc định: tối) ----\nfunction initTheme() {\n  const saved = localStorage.getItem(\'theme\');\n  if (saved !== \'light\') document.body.classList.add(\'dark\');\n  updateThemeIcon();\n}\nfunction toggleTheme() {\n  document.body.classList.toggle(\'dark\');\n  localStorage.setItem(\'theme\', document.body.classList.contains(\'dark\') ? \'dark\' : \'light\');\n  updateThemeIcon();\n}\nfunction updateThemeIcon() {\n  document.getElementById(\'themeToggle\').textContent = document.body.classList.contains(\'dark\') ? \'☀️\' : \'🌙\';\n}\n\n// ---- Tab switching (sidebar) ----\ndocument.querySelectorAll(\'.navitem\').forEach(el => {\n  el.addEventListener(\'click\', () => {\n    document.querySelectorAll(\'.navitem\').forEach(n => n.classList.remove(\'active\'));\n    document.querySelectorAll(\'.tab-section\').forEach(s => s.classList.remove(\'active\'));\n    el.classList.add(\'active\');\n    document.getElementById(\'tab-\' + el.dataset.tab).classList.add(\'active\');\n    if (el.dataset.tab === \'library\' || el.dataset.tab === \'tts\') loadVoices();\n  });\n});\n\n// ---- Right-panel segmented tabs (Giọng nói / Lịch sử) ----\nfunction switchRightTab(tab) {\n  document.getElementById(\'segVoices\').classList.toggle(\'active\', tab === \'voices\');\n  document.getElementById(\'segHistory\').classList.toggle(\'active\', tab === \'history\');\n  document.getElementById(\'rightPanel-voices\').style.display = tab === \'voices\' ? \'block\' : \'none\';\n  document.getElementById(\'rightPanel-history\').style.display = tab === \'history\' ? \'block\' : \'none\';\n}\n\nfunction setStatus(id, msg, cls) {\n  const el = document.getElementById(id);\n  el.textContent = msg || \'\';\n  el.className = \'status\' + (cls ? \' \' + cls : \'\');\n}\n\n// ---- Escape text người dùng nhập trước khi chèn vào innerHTML (chống XSS) ----\nfunction escapeHtml(str) {\n  return String(str ?? \'\').replace(/[&<>"\']/g, ch => ({\n    \'&\': \'&amp;\', \'<\': \'&lt;\', \'>\': \'&gt;\', \'"\': \'&quot;\', "\'": \'&#39;\'\n  }[ch]));\n}\n\n// ---- Lấy thông báo lỗi dễ đọc từ response lỗi của FastAPI (JSON {"detail": "..."}) ----\nasync function extractErrorMessage(response) {\n  try {\n    const data = await response.json();\n    if (data && data.detail) return data.detail;\n  } catch (e) { /* không phải JSON */ }\n  return \'HTTP \' + response.status;\n}\n\n// ---- Character / word counters + cảnh báo độ dài ----\nfunction updateCounts() {\n  const text = document.getElementById(\'ttsText\').value;\n  document.getElementById(\'charCount\').textContent = text.length + \' ký tự\';\n  const words = text.trim().length ? text.trim().split(/\\s+/).length : 0;\n  document.getElementById(\'wordCount\').textContent = words + \' từ\';\n\n  const warnEl = document.getElementById(\'lengthWarning\');\n  const genBtn = document.getElementById(\'genBtn\');\n  if (words > HARD_WORD_LIMIT) {\n    warnEl.textContent = `Văn bản vượt quá giới hạn ${HARD_WORD_LIMIT} từ (hiện tại: ${words} từ) — vui lòng rút ngắn bớt.`;\n    warnEl.className = \'status err\';\n    genBtn.disabled = true;\n  } else if (words > SOFT_WORD_LIMIT) {\n    warnEl.textContent = `Văn bản khá dài (${words} từ) — sẽ được xử lý theo từng đoạn nhỏ và có thể mất vài phút.`;\n    warnEl.className = \'status warn\';\n    genBtn.disabled = false;\n  } else {\n    warnEl.textContent = \'\';\n    warnEl.className = \'status\';\n    genBtn.disabled = false;\n  }\n}\n\n// ---- Filename helper: "voice-<ngay-gio>.wav" ----\nfunction makeFilename(text, ext) {\n  ext = ext || \'wav\';\n  const now = new Date();\n  const pad = n => String(n).padStart(2, \'0\');\n  const stamp = `${now.getFullYear()}${pad(now.getMonth()+1)}${pad(now.getDate())}-${pad(now.getHours())}${pad(now.getMinutes())}${pad(now.getSeconds())}`;\n  return `voice-${stamp}.${ext}`;\n}\n\n// ---- Load saved voice profiles ----\nasync function loadVoices() {\n  try {\n    const r = await fetch(BASE + \'/profiles\');\n    if (!r.ok) throw new Error(\'HTTP \' + r.status);\n    allProfiles = await r.json();\n\n    if (selectedProfileId && !allProfiles.some(p => p.id === selectedProfileId)) {\n      selectedProfileId = "";\n      selectedProfileName = "Mặc định";\n      document.getElementById(\'selectedVoiceName\').textContent = selectedProfileName;\n    }\n\n    renderVoicePicker();\n\n    const listEl = document.getElementById(\'voiceList\');\n    if (allProfiles.length === 0) {\n      listEl.innerHTML = \'<div class="empty">Chưa có giọng nào — sang tab "Giọng nói của bạn" để tạo giọng đầu tiên.</div>\';\n    } else {\n      listEl.innerHTML = `<div class="counts" style="margin-bottom:10px">${allProfiles.length}/${MAX_PROFILES} giọng đã lưu</div>`;\n      allProfiles.forEach(p => {\n        const div = document.createElement(\'div\');\n        div.className = \'voice-item\';\n        div.innerHTML = `<div style="display:flex;align-items:center;gap:10px">\n                            <button class="preview-btn" id="prevbtn-${p.id}" onclick="previewVoice(\'${p.id}\', this)" title="Nghe thử">▷</button>\n                            <div><div class="name">${escapeHtml(p.name)}</div><div class="meta">${escapeHtml(p.kind || \'clone\')}</div></div>\n                          </div>\n                          <div class="actions">\n                            <button class="btn secondary" onclick="useVoice(\'${p.id}\')">Dùng giọng này</button>\n                            <button class="btn danger" onclick="deleteVoice(\'${p.id}\', this)">Xoá</button>\n                          </div>`;\n        listEl.appendChild(div);\n      });\n    }\n  } catch (e) {\n    console.error(e);\n  }\n}\n\n// ---- Voice picker (right panel of "Đọc văn bản") ----\nfunction renderVoicePicker() {\n  const q = (document.getElementById(\'voiceSearch\').value || \'\').trim().toLowerCase();\n  const listEl = document.getElementById(\'voicePickerList\');\n  listEl.innerHTML = \'\';\n\n  const items = [{ id: \'\', name: \'Mặc định (tự động)\', kind: \'system\' }, ...allProfiles];\n  const filtered = items.filter(p => p.name.toLowerCase().includes(q));\n\n  if (filtered.length === 0) {\n    listEl.innerHTML = \'<div class="empty">Không tìm thấy giọng nào.</div>\';\n    return;\n  }\n\n  filtered.forEach(p => {\n    const row = document.createElement(\'div\');\n    row.className = \'voice-row\' + (p.id === selectedProfileId ? \' selected\' : \'\');\n    row.onclick = () => selectVoice(p.id, p.name);\n    const playIcHtml = p.id\n      ? `<div class="play-ic" id="pickpic-${p.id}" onclick="previewVoice(\'${p.id}\', this, event)">▷</div>`\n      : `<div class="play-ic">•</div>`;\n    row.innerHTML = `${playIcHtml}\n                      <div><div class="vname">${escapeHtml(p.name)}</div>\n                      <div class="vmeta">${p.id ? \'Giọng do bạn tự tạo\' : \'Giọng mặc định của hệ thống\'}</div></div>`;\n    listEl.appendChild(row);\n  });\n}\n\n// ---- Nghe thử giọng đã lưu (dùng chung 1 thẻ audio ẩn) ----\nconst libPreviewAudio = document.getElementById(\'libPreviewAudio\');\nlet previewingId = null;\n\nfunction previewVoice(id, btnEl, event) {\n  if (event) event.stopPropagation();\n  const isSameAndPlaying = previewingId === id && !libPreviewAudio.paused;\n  document.querySelectorAll(\'.preview-btn.playing, .play-ic.playing\').forEach(el => el.classList.remove(\'playing\'));\n  libPreviewAudio.pause();\n\n  if (isSameAndPlaying) { previewingId = null; return; }\n\n  previewingId = id;\n  libPreviewAudio.src = BASE + \'/profiles/\' + id + \'/preview\';\n  libPreviewAudio.play().then(() => {\n    if (btnEl) btnEl.classList.add(\'playing\');\n  }).catch(() => {\n    previewingId = null;\n    alert(\'Giọng này chưa có bản nghe thử (được tạo trước khi có tính năng này).\');\n  });\n}\nlibPreviewAudio.addEventListener(\'ended\', () => {\n  document.querySelectorAll(\'.preview-btn.playing, .play-ic.playing\').forEach(el => el.classList.remove(\'playing\'));\n  previewingId = null;\n});\n\nfunction selectVoice(id, name) {\n  selectedProfileId = id;\n  selectedProfileName = name;\n  document.getElementById(\'selectedVoiceName\').textContent = name;\n  renderVoicePicker();\n}\n\nfunction useVoice(id) {\n  const p = allProfiles.find(x => x.id === id);\n  document.querySelector(\'.navitem[data-tab="tts"]\').click();\n  selectVoice(id, p ? p.name : id);\n  switchRightTab(\'voices\');\n}\n\nasync function deleteVoice(id, btnEl) {\n  if (!confirm(\'Xoá giọng nói này khỏi thư viện?\')) return;\n  btnEl.disabled = true;\n  try {\n    const r = await fetch(BASE + \'/profiles/\' + id, { method: \'DELETE\' });\n    if (!r.ok) throw new Error(\'HTTP \' + r.status);\n    loadVoices();\n  } catch (e) {\n    alert(\'Không xoá được (backend có thể chưa hỗ trợ xoá qua API): \' + e.message);\n    btnEl.disabled = false;\n  }\n}\n\n// ---- Recording ----\nasync function toggleRecord() {\n  if (!isRecording) {\n    try {\n      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });\n      recordedChunks = [];\n      mediaRecorder = new MediaRecorder(stream);\n      mediaRecorder.ondataavailable = e => recordedChunks.push(e.data);\n      mediaRecorder.onstop = () => {\n        recordedBlob = new Blob(recordedChunks, { type: \'audio/webm\' });\n        const preview = document.getElementById(\'refPreview\');\n        preview.src = URL.createObjectURL(recordedBlob);\n        preview.style.display = \'block\';\n        document.getElementById(\'saveVoiceBtn\').disabled = false;\n        stream.getTracks().forEach(t => t.stop());\n      };\n      mediaRecorder.start();\n      isRecording = true;\n      document.getElementById(\'recBtn\').innerHTML = \'<span class="rec-dot"></span>Dừng ghi âm\';\n      setStatus(\'recStatus\', \'Đang ghi âm... đọc to rõ khoảng 5-10 giây.\');\n    } catch (e) {\n      setStatus(\'recStatus\', \'Không truy cập được microphone: \' + e.message, \'err\');\n    }\n  } else {\n    mediaRecorder.stop();\n    isRecording = false;\n    document.getElementById(\'recBtn\').innerHTML = \'🎙️ Bắt đầu ghi âm\';\n    setStatus(\'recStatus\', \'Đã ghi âm xong — nghe lại bên dưới rồi bấm Lưu.\', \'ok\');\n  }\n}\nfunction onFileChosen() {\n  const f = document.getElementById(\'fileInput\').files[0];\n  if (!f) return;\n  recordedBlob = f;\n  const preview = document.getElementById(\'refPreview\');\n  preview.src = URL.createObjectURL(f);\n  preview.style.display = \'block\';\n  document.getElementById(\'saveVoiceBtn\').disabled = false;\n  setStatus(\'recStatus\', \'Đã chọn file: \' + f.name, \'ok\');\n}\n\nasync function saveVoiceProfile() {\n  const name = document.getElementById(\'voiceName\').value.trim();\n  if (!name) { setStatus(\'voiceStatus\', \'Nhập tên giọng nói trước đã.\', \'err\'); return; }\n  if (!recordedBlob) { setStatus(\'voiceStatus\', \'Chưa có audio mẫu.\', \'err\'); return; }\n  const btn = document.getElementById(\'saveVoiceBtn\');\n  btn.disabled = true;\n  setStatus(\'voiceStatus\', \'Đang lưu giọng nói...\');\n  try {\n    const fd = new FormData();\n    fd.append(\'name\', name);\n    fd.append(\'kind\', \'clone\');\n    fd.append(\'ref_audio\', recordedBlob, \'reference.wav\');\n    const r = await fetch(BASE + \'/profiles\', { method: \'POST\', body: fd });\n    if (!r.ok) throw new Error(await extractErrorMessage(r));\n    setStatus(\'voiceStatus\', \'Đã lưu giọng "\' + name + \'" vào thư viện.\', \'ok\');\n    document.getElementById(\'voiceName\').value = \'\';\n    recordedBlob = null;\n    document.getElementById(\'refPreview\').style.display = \'none\';\n    loadVoices();\n  } catch (e) {\n    setStatus(\'voiceStatus\', \'Lưu thất bại: \' + e.message, \'err\');\n  } finally {\n    btn.disabled = false;\n  }\n}\n\n// ---- TTS generation (job + polling — xử lý theo từng đoạn cho văn bản dài) ----\nfunction sleep(ms) { return new Promise(res => setTimeout(res, ms)); }\n\nasync function generateTTS() {\n  const text = document.getElementById(\'ttsText\').value.trim();\n  if (!text) { setStatus(\'ttsStatus\', \'Nhập văn bản trước đã.\', \'err\'); return; }\n  const words = text.split(/\\s+/).length;\n  if (words > HARD_WORD_LIMIT) { setStatus(\'ttsStatus\', `Văn bản vượt quá ${HARD_WORD_LIMIT} từ.`, \'err\'); return; }\n\n  const btn = document.getElementById(\'genBtn\');\n  btn.disabled = true;\n  setStatus(\'ttsStatus\', \'Đang gửi văn bản...\');\n  try {\n    const fd = new FormData();\n    fd.append(\'text\', text);\n    if (selectedProfileId) fd.append(\'profile_id\', selectedProfileId);\n    const r = await fetch(BASE + \'/generate\', { method: \'POST\', body: fd });\n    if (!r.ok) throw new Error(await extractErrorMessage(r));\n    const { job_id, total_chunks } = await r.json();\n\n    // Poll trạng thái tới khi xong\n    let status = null;\n    while (true) {\n      const sr = await fetch(BASE + \'/generate/\' + job_id + \'/status\');\n      if (!sr.ok) throw new Error(\'HTTP \' + sr.status);\n      status = await sr.json();\n      if (status.status === \'done\' || status.status === \'error\') break;\n      setStatus(\'ttsStatus\', total_chunks > 1\n        ? `Đang tạo giọng nói... (đoạn ${status.done}/${status.total})`\n        : \'Đang tạo giọng nói...\');\n      await sleep(900);\n    }\n    if (status.status === \'error\') throw new Error(status.error || \'Lỗi không xác định\');\n\n    const ar = await fetch(BASE + \'/generate/\' + job_id + \'/audio\');\n    if (!ar.ok) throw new Error(\'HTTP \' + ar.status);\n    const blob = await ar.blob();\n    const url = URL.createObjectURL(blob);\n    const filename = makeFilename(text);\n    const genTime = ar.headers.get(\'X-Gen-Time\');\n\n    setStatus(\'ttsStatus\', genTime ? `Xong trong ${genTime}s.` : \'Xong.\', \'ok\');\n    showPlayer(url, filename, selectedProfileName, genTime);\n    addHistory(text, selectedProfileName, url, filename, genTime);\n  } catch (e) {\n    setStatus(\'ttsStatus\', \'Tạo thất bại: \' + e.message, \'err\');\n  } finally {\n    btn.disabled = false;\n  }\n}\n\nfunction addHistory(text, voiceName, url, filename, genTime) {\n  historyItems.unshift({ text, voiceName, url, filename, genTime, time: new Date().toLocaleTimeString(\'vi-VN\') });\n  renderHistory();\n}\nfunction renderHistory() {\n  const fullEl = document.getElementById(\'historyList\');\n  const panelEl = document.getElementById(\'historyPanelList\');\n\n  if (historyItems.length === 0) {\n    fullEl.innerHTML = \'<div class="empty">Chưa có lịch sử nào.</div>\';\n    panelEl.innerHTML = \'<div class="empty">Chưa có lịch sử nào.</div>\';\n    return;\n  }\n\n  fullEl.innerHTML = \'\';\n  historyItems.forEach(h => {\n    const div = document.createElement(\'div\');\n    div.className = \'hist-item\';\n    div.innerHTML = `<div class="text">${escapeHtml(h.text.length > 120 ? h.text.slice(0,120)+\'…\' : h.text)}</div>\n                      <div class="meta">${escapeHtml(h.voiceName)} · ${escapeHtml(h.time)}</div>\n                      <audio controls src="${h.url}"></audio>\n                      <div class="row">\n                        <a href="${h.url}" download="${h.filename}" class="btn secondary">⬇️ Tải xuống</a>\n                      </div>`;\n    fullEl.appendChild(div);\n  });\n\n  panelEl.innerHTML = \'\';\n  historyItems.forEach(h => {\n    const row = document.createElement(\'div\');\n    row.className = \'hist-row\';\n    row.onclick = () => showPlayer(h.url, h.filename, h.voiceName, h.genTime);\n    row.innerHTML = `<div class="htext">${escapeHtml(h.text.length > 40 ? h.text.slice(0,40)+\'…\' : h.text)}</div>\n                      <div class="hmeta">${escapeHtml(h.voiceName)} · ${escapeHtml(h.time)}</div>`;\n    panelEl.appendChild(row);\n  });\n}\n\n// ---- Bottom player bar ----\nconst playerAudio = document.getElementById(\'playerAudio\');\nlet currentSpeed = 1;\n\nfunction fmtTime(s) {\n  if (!isFinite(s) || s < 0) s = 0;\n  const m = Math.floor(s / 60);\n  const sec = Math.floor(s % 60);\n  return m + \':\' + String(sec).padStart(2, \'0\');\n}\n\nfunction showPlayer(url, filename, voiceName, genTime) {\n  playerAudio.src = url;\n  playerAudio.playbackRate = currentSpeed;\n  document.getElementById(\'playerName\').textContent = filename;\n  document.getElementById(\'playerMeta\').textContent = voiceName + (genTime ? \' · \' + genTime + \'s\' : \'\');\n  document.getElementById(\'playerDownload\').href = url;\n  document.getElementById(\'playerDownload\').download = filename;\n  document.getElementById(\'playerBar\').style.display = \'flex\';\n  playerAudio.play();\n}\n\nfunction togglePlayPause() {\n  if (!playerAudio.src) return;\n  if (playerAudio.paused) playerAudio.play(); else playerAudio.pause();\n}\nfunction stopPlayer() {\n  if (!playerAudio.src) return;\n  playerAudio.pause();\n  playerAudio.currentTime = 0;\n}\nfunction cycleSpeed() {\n  const speeds = [1, 1.25, 1.5, 2, 0.75];\n  const idx = speeds.indexOf(currentSpeed);\n  currentSpeed = speeds[(idx + 1) % speeds.length];\n  playerAudio.playbackRate = currentSpeed;\n  document.getElementById(\'speedBtn\').textContent = currentSpeed + \'x\';\n}\nfunction toggleMute() {\n  playerAudio.muted = !playerAudio.muted;\n  document.getElementById(\'muteBtn\').textContent = playerAudio.muted ? \'🔇\' : \'🔊\';\n}\nfunction seekPlayer() {\n  if (!playerAudio.duration) return;\n  const pct = document.getElementById(\'playerSeek\').value;\n  playerAudio.currentTime = (pct / 100) * playerAudio.duration;\n}\nfunction closePlayer() {\n  playerAudio.pause();\n  document.getElementById(\'playerBar\').style.display = \'none\';\n}\n\nplayerAudio.addEventListener(\'play\', () => { document.getElementById(\'playPauseBtn\').textContent = \'⏸\'; });\nplayerAudio.addEventListener(\'pause\', () => { document.getElementById(\'playPauseBtn\').textContent = \'▷\'; });\nplayerAudio.addEventListener(\'timeupdate\', () => {\n  document.getElementById(\'playerTime\').textContent = fmtTime(playerAudio.currentTime) + \' / \' + fmtTime(playerAudio.duration);\n  const seek = document.getElementById(\'playerSeek\');\n  if (playerAudio.duration) {\n    const pct = (playerAudio.currentTime / playerAudio.duration) * 100;\n    seek.value = pct;\n    seek.style.setProperty(\'--pct\', pct + \'%\');\n  }\n});\nplayerAudio.addEventListener(\'loadedmetadata\', () => {\n  document.getElementById(\'playerTime\').textContent = fmtTime(0) + \' / \' + fmtTime(playerAudio.duration);\n});\n\n// ---- Giữ phiên sống + báo khi mất kết nối backend ----\nlet backendUp = true;\nasync function heartbeat() {\n  try {\n    const r = await fetch(BASE + \'/health\', { cache: \'no-store\' });\n    if (!r.ok) throw new Error(\'bad status\');\n    if (!backendUp) { loadVoices(); }  // vừa kết nối lại -> nạp lại dữ liệu\n    backendUp = true;\n    document.getElementById(\'connBanner\').style.display = \'none\';\n  } catch (e) {\n    backendUp = false;\n    document.getElementById(\'connBanner\').style.display = \'block\';\n  }\n}\nsetInterval(heartbeat, 60000);  // mỗi 60s — vừa giữ tab "có hoạt động", vừa dò kết nối\n\ninitTheme();\nupdateCounts();\nloadVoices();\nheartbeat();\n</script>\n</body>\n</html>\n'

with open(os.path.join(FRONTEND_DIR, "index.html"), "w", encoding="utf-8") as f:
    f.write(UI_HTML)
print("Đã ghi giao diện web.")


In [ ]:
# 4. Mount Google Drive (tuỳ chọn) — lưu bền vững giọng nói + model đã tải
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/omnispeak_data"
HF_DIR = "/content/drive/MyDrive/omnispeak_hf_cache"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
os.environ["OMNISPEAK_DATA_DIR"] = DATA_DIR
os.environ["HF_HOME"] = HF_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"  # Drive (FUSE) không hỗ trợ symlink
print("Giọng nói + model sẽ lưu bền vững trên Drive.")

In [ ]:
# 5. Tải trước model (bỏ qua cũng được — sẽ tự tải khi khởi động backend)
from huggingface_hub import snapshot_download

print("Model tại:", snapshot_download("k2-fsa/OmniVoice"))


In [ ]:
# 6. Backend API — FastAPI gọi thẳng OmniVoice
import os

APP_DIR = "/content/omnispeak_app"
os.makedirs(APP_DIR, exist_ok=True)

BACKEND_PY = '''\
import io
import json
import os
import time
import uuid
from pathlib import Path
from typing import Optional

import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, Form, UploadFile, File, HTTPException
from fastapi.responses import Response
from fastapi.staticfiles import StaticFiles
from omnivoice import OmniVoice, VoiceClonePrompt

DATA_DIR = Path(os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data"))
PROFILES_DIR = DATA_DIR / "profiles"
INDEX_PATH = DATA_DIR / "profiles.json"
FRONTEND_DIR = os.environ.get("OMNISPEAK_FRONTEND_DIR", "/content/omnispeak_frontend")
PROFILES_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
MODEL = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype, load_asr=True)


def _load_index():
    return json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else []


def _save_index(items):
    INDEX_PATH.write_text(json.dumps(items, ensure_ascii=False, indent=2))


@app.get("/health")
def health():
    return {"status": "ok", "device": "cuda" if torch.cuda.is_available() else "cpu"}


@app.post("/generate")
async def generate(text: str = Form(...), profile_id: Optional[str] = Form(None)):
    t0 = time.time()
    kwargs = {}
    if profile_id:
        p = PROFILES_DIR / f"{profile_id}.pt"
        if not p.exists():
            raise HTTPException(404, f"Profile {profile_id} not found")
        kwargs["voice_clone_prompt"] = VoiceClonePrompt.load(str(p))

    audio = MODEL.generate(text=text, **kwargs)
    gen_time = time.time() - t0

    # model.generate() có thể trả numpy hoặc torch — chuẩn hoá về numpy mono
    wav = audio[0]
    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().float().numpy()
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.T
        if wav.shape[1] == 1:
            wav = wav[:, 0]

    buf = io.BytesIO()
    sf.write(buf, wav, 24000, format="WAV", subtype="PCM_16")
    return Response(content=buf.getvalue(), media_type="audio/wav",
                     headers={"X-Gen-Time": f"{gen_time:.2f}"})


@app.get("/profiles")
def list_profiles():
    return _load_index()


@app.post("/profiles")
async def create_profile(name: str = Form(...), kind: str = Form("clone"),
                          ref_audio: UploadFile = File(...), ref_text: Optional[str] = Form(None)):
    items = _load_index()
    existing = next((p for p in items if p["name"] == name), None)
    if existing:
        return existing

    profile_id = uuid.uuid4().hex[:12]
    tmp = DATA_DIR / f"_upload_{profile_id}.wav"
    tmp.write_bytes(await ref_audio.read())
    try:
        prompt = MODEL.create_voice_clone_prompt(ref_audio=str(tmp), ref_text=ref_text or None)
        prompt.save(str(PROFILES_DIR / f"{profile_id}.pt"))
    finally:
        tmp.unlink(missing_ok=True)

    entry = {"id": profile_id, "name": name, "kind": kind}
    items.append(entry)
    _save_index(items)
    return entry


@app.delete("/profiles/{profile_id}")
def delete_profile(profile_id: str):
    items = _load_index()
    remaining = [p for p in items if p["id"] != profile_id]
    if len(remaining) == len(items):
        raise HTTPException(404, "Not found")
    _save_index(remaining)
    (PROFILES_DIR / f"{profile_id}.pt").unlink(missing_ok=True)
    return {"deleted": profile_id}


app.mount("/", StaticFiles(directory=FRONTEND_DIR, html=True), name="frontend")
'''

with open(os.path.join(APP_DIR, "backend.py"), "w", encoding="utf-8") as f:
    f.write(BACKEND_PY)
print("Đã ghi backend.py.")


In [ ]:
# 7. Khởi động backend
FORCE_RESTART = True  # luôn nạp lại code mới nhất từ cell 6

import json
import os
import subprocess
import sys
import time
import urllib.request

APP_DIR = "/content/omnispeak_app"
PORT = 3900
LOG_PATH = "/content/omnispeak_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = None if FORCE_RESTART else health()
if info:
    print("Backend đang chạy —", info)
else:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    env["OMNISPEAK_DATA_DIR"] = os.environ.get("OMNISPEAK_DATA_DIR", "/content/omnispeak_data")
    env["OMNISPEAK_FRONTEND_DIR"] = "/content/omnispeak_frontend"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend:app", "--app-dir", APP_DIR,
         "--host", "127.0.0.1", "--port", str(PORT)],
        env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Đang khởi động (PID {proc.pid})...")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print("Backend đã sẵn sàng —", info)
    else:
        try:
            tail = "".join(open(LOG_PATH, errors="replace").readlines()[-40:])
        except OSError:
            tail = "(không có log)"
        raise SystemExit(f"Backend không lên được sau 5 phút.\n--- log ---\n{tail}")


In [ ]:
# 8. Mở giao diện web
from google.colab import output

output.serve_kernel_port_as_window(3900)


### Tổng kết

Danh sách giọng nói đã lưu.

In [ ]:
# Tổng kết
import requests

try:
    profiles = requests.get("http://127.0.0.1:3900/profiles", timeout=15).json()
    print(f"Giọng đã lưu ({len(profiles)}):")
    for p in profiles:
        print(" ", p["id"], p.get("name"))
except Exception as e:
    print("Không lấy được danh sách:", e)


## Xử lý sự cố

- **`device: cpu` hoặc generate chậm** — bật GPU: Runtime → Change runtime type → T4 GPU.
- **Sửa `backend.py` (cell 6) xong mà lỗi vẫn y nguyên** — cell 7 có `FORCE_RESTART = True`, chạy lại cell 7 là tự nạp code mới, không cần tự kill process.
- **Muốn giữ giọng nói + model qua các phiên sau** — chạy cell 4 (mount Drive) trước cell 5.
- **Tab UI trắng hoặc lỗi** — chạy lại cell 7 rồi cell 8. Cho phép pop-up cho `colab.research.google.com`; hoặc đổi cell 8 sang `output.serve_kernel_port_as_iframe(3900)` để nhúng UI ngay trong notebook.
- **Xem log backend** — `/content/omnispeak_backend.log`.
